# 🧭 Two microstructures, two relaxation times — `carreau_carreau` in the browser

*The same workflow as the [rheofit case study](https://rheofit.readthedocs.io/en/latest/walkthrough-carreau-carreau.html),
run live in JupyterLite. A mixed wormlike-micelle + polymer surfactant system shows (at least)
two relaxation times — one Carreau model cannot bend twice, but the microstructure-informed
`carreau_carreau` sum can.*

**Run each cell in order** (▶). The first cell installs `rheofit` in your browser — no server involved.

In [ ]:
%pip install -q rheofit

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import rheofit
print("rheofit", rheofit.__version__ if hasattr(rheofit, "__version__") else "installed")

## 🧪 The dataset

`aos_2_1.json`: equilibrium flow curves of a mixed surfactant system — **wormlike micelles (WLM)
plus a polymer solution**. Seven sweeps: **18, 20, 22, 24, 26, 28 °C, plus a repeat at 18 °C**.

In [ ]:
# the data file ships next to this notebook
DATA = next(p for p in [Path("aos_2_1.json"), Path("mirm/aos_2_1.json"),
                        Path("content/mirm/aos_2_1.json")] if p.exists())
print("using:", DATA)
rheofit.print_steps(str(DATA))

temps = []
for i in range(7):
    d = rheofit.load_step(str(DATA), i)
    temps.append(round(float(d["Temperature / degC"].mean()), 1))
print("temperatures / °C:", temps)

## 🤔 The modeling assumption

Decouple the contributions: **WLM → stress plateau → Carreau with n = 0**;
**polymer → Carreau with n = 0.5**. That is exactly `carreau_carreau`:

σ = η₀₁·γ̇·[1 + (λ₁·γ̇)²]^(−¼) + η₀₂·γ̇·[1 + (λ₂·γ̇)²]^(−½)

## 📉 One Carreau is not enough — 18 °C

In [ ]:
df18 = rheofit.load_step(str(DATA), 0)  # 18 °C
x = df18["Shear rate / 1/s"].to_numpy()
y = df18["Stress / Pa"].to_numpy()

r1 = rheofit.fit(df18, "carreau", effort="thorough", seed=0)
p1 = {k: v["value"] for k, v in r1["params"].items()}
print("single Carreau: n = %.3f, RedChi2 = %.3e" % (p1["n"], r1["redchi"]))
g = np.logspace(-2, 2, 300)
s_c = p1["eta_0"]*g*(1 + (p1["lambda_val"]*g)**2)**((p1["n"]-1)/2)

fig, (a1, a2) = plt.subplots(2, 1, figsize=(7, 6), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1]})
a1.loglog(x, y, "ko", ms=4, label="data (18 °C)")
a1.loglog(g, s_c, "b-", lw=2, label="single Carreau (n=%.2f)" % p1["n"])
a1.set_ylabel("stress / Pa"); a1.legend(); a1.grid(True, which="both", alpha=0.3)
a1.set_title("One relaxation time is not enough at 18 °C")
resid = (y - p1["eta_0"]*x*(1 + (p1["lambda_val"]*x)**2)**((p1["n"]-1)/2)) / y * 100
a2.semilogx(x, resid, "bo", ms=4)
a2.axhline(0, color="k", lw=0.8)
a2.set_xlabel("shear rate / 1/s"); a2.set_ylabel("rel. resid. / %")
a2.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

The single Carreau settles on a **compromise exponent n ≈ 0.61** — halfway between the WLM
plateau mode (n = 0) and the polymer mode (n = 0.5) — and the residuals trace a systematic S-shape.

## ✅ `carreau_carreau` at 18 °C

In [ ]:
r2 = rheofit.fit(df18, "carreau_carreau", effort="thorough", seed=0)
p2 = {k: v["value"] for k, v in r2["params"].items()}
print("RedChi2 = %.3e (%.0fx better), cond = %.1f" % (r2["redchi"], r1["redchi"]/r2["redchi"], r2["cond"]))
for k, v in r2["params"].items():
    print("%-12s = %10.4g  ± %.1f%%" % (k, v["value"], 100*v["stderr"]/v["value"]))

e01, l1, e02, l2 = p2["eta_0_1"], p2["lambda_val_1"], p2["eta_0_2"], p2["lambda_val_2"]
s1 = e01*g*(1 + (l1*g)**2)**(-0.25)   # polymer, n = 0.5
s2 = e02*g*(1 + (l2*g)**2)**(-0.5)    # WLM, n = 0 -> stress plateau

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(x, y, "ko", ms=4, label="data (18 °C)")
ax.loglog(g, s1+s2, "r-", lw=2, label="carreau–carreau fit")
ax.loglog(g, s1, "b--", lw=1.5, label="polymer term (n=0.5)")
ax.loglog(g, s2, "g--", lw=1.5, label="WLM term (n=0, plateau)")
ax.set_xlabel("shear rate / 1/s"); ax.set_ylabel("stress / Pa")
ax.set_title("Carreau–Carreau decomposition at 18 °C")
ax.legend(); ax.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

Two relaxation times ~90× apart, every parameter identified — a slow polymer mode plus a fast
WLM mode flattening into its stress plateau.

## 🌡️ Across temperatures

Fit every sweep (lighter effort here to keep the browser happy) and watch the parameters move.

In [ ]:
rows = []
for i, T in enumerate(temps):
    df = rheofit.load_step(str(DATA), i)
    r = rheofit.fit(df, "carreau_carreau", effort="normal", seed=0)
    q = {k: v["value"] for k, v in r["params"].items()}
    rows.append((T, q["eta_0_1"], q["lambda_val_1"], q["eta_0_2"], q["lambda_val_2"], r["redchi"]))
    print("T=%4.1f °C  eta01=%6.2f  lam1=%5.2f  eta02=%5.2f  lam2=%.4f  RedChi2=%.2e"
          % rows[-1])
res = pd.DataFrame(rows, columns=["T", "eta01", "lam1", "eta02", "lam2", "redchi"])

fig, axs = plt.subplots(2, 2, figsize=(8, 6), sharex=True)
for ax, col, lab in zip(axs.ravel(), ["eta01", "lam1", "eta02", "lam2"],
        ["η₀ polymer / Pa·s", "λ polymer / s", "η₀ WLM / Pa·s", "λ WLM / s"]):
    ax.semilogy(res["T"], res[col], "o-")
    ax.set_ylabel(lab); ax.grid(True, which="both", alpha=0.3)
for ax in axs[1]: ax.set_xlabel("temperature / °C")
fig.suptitle("Carreau–Carreau parameters vs temperature")
plt.tight_layout(); plt.show()

## 💡 Takeaway

Both relaxation times shorten with temperature and both viscosities fall — the expected thermal
softening, cleanly separated per microstructure. A single-mode model forced onto such data returns
a compromise; the microstructure-informed sum fits the physics instead of averaging it.

*Full write-up: the [rheofit case study](https://rheofit.readthedocs.io/en/latest/walkthrough-carreau-carreau.html)
(runs `effort="thorough"` everywhere and adds the 18 °C repeat).*
